# Knowledge Distillation on the Beans Dataset - Evaluation & Testing Notebook
This independent notebook evaluates performance using **only** the `test` split to eliminate data leakage. It benchmarks the Teacher, the Initial Untrained Student, and the Distilled Student before demonstrating visual inferences.

In [ ]:
# Re-initialize Environment & Test Split (No Overlap)
!pip install transformers datasets accelerate evaluate matplotlib --upgrade

import torch
import evaluate
import numpy as np
import random
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoImageProcessor, AutoModelForImageClassification, MobileNetV2Config, MobileNetV2ForImageClassification, Trainer, TrainingArguments, DefaultDataCollator

# Load dataset and extract splits
dataset = load_dataset("AI-Lab-Makerere/beans")
teacher_processor = AutoImageProcessor.from_pretrained("merve/beans-vit-224")

def process(examples):
    return teacher_processor(examples["image"])

processed_datasets = dataset.map(process, batched=True)
num_labels = len(processed_datasets["test"].features["labels"].names)
data_collator = DefaultDataCollator()

accuracy = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    acc = accuracy.compute(references=labels, predictions=np.argmax(predictions, axis=1))
    return {"accuracy": acc["accuracy"]}

# Setup training arguments structure to maintain FP16 alignment flags across evaluations
eval_args = TrainingArguments(
    output_dir="eval-logs",
    fp16=True,
    report_to="none"
)

In [ ]:
# Step-by-Step Benchmarking Over the Test Split
print("Evaluating Baseline Models on Test Split...")

# 1. Teacher Model Performance Check
teacher_model = AutoModelForImageClassification.from_pretrained("merve/beans-vit-224", num_labels=num_labels)
teacher_eval_trainer = Trainer(
    model=teacher_model,
    args=eval_args,
    eval_dataset=processed_datasets["test"],  # Test set only
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
teacher_results = teacher_eval_trainer.evaluate()
teacher_accuracy = teacher_results['eval_accuracy'] * 100

# 2. Student Before KD Performance Check
student_config = MobileNetV2Config()
student_config.num_labels = num_labels
untrained_student = MobileNetV2ForImageClassification(student_config)

student_before_trainer = Trainer(
    model=untrained_student,
    args=eval_args,
    eval_dataset=processed_datasets["test"],  # Test set only
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
before_results = student_before_trainer.evaluate()
student_before_accuracy = before_results['eval_accuracy'] * 100

# 3. Load Trained Student from disk (After KD)
try:
    distilled_student = MobileNetV2ForImageClassification.from_pretrained("./saved_distilled_student")
    student_after_trainer = Trainer(
        model=distilled_student,
        args=eval_args,
        eval_dataset=processed_datasets["test"],  # Test set only
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    after_results = student_after_trainer.evaluate()
    student_after_accuracy = after_results['eval_accuracy'] * 100
except OSError:
    print("\n[ERROR]: Missing weights file! Run training.ipynb first to save the model.")
    student_after_accuracy = 0.0

In [ ]:
# Display Metrics Comparison Table
print("\n" + "="*60)
print(f"{'Model Setup (Evaluated on Test Split Only)':<42} | {'Test Accuracy':<15}")
print("="*60)
print(f"{'1. Teacher (ViT-base)':<42} | {teacher_accuracy:.2f}%")
print(f"{'2. Student Before KD (MobileNetV2)':<42} | {student_before_accuracy:.2f}%")
print(f"{'3. Student After KD (MobileNetV2)':<42} | {student_after_accuracy:.2f}%")
print("="*60)

In [ ]:
# Sample Visualization (Inference Test)
labels = dataset["test"].features["labels"].names
distilled_student.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
distilled_student.to(device)

samples = []
for i in range(len(dataset["test"])):
    sample = dataset["test"][i]
    image = sample["image"]
    true_idx = sample["labels"]

    inputs = teacher_processor(image, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = distilled_student(**inputs)

    pred_idx = outputs.logits.argmax(-1).item()
    status = "✓ Correct" if pred_idx == true_idx else "✗ Wrong"

    samples.append({
        "image": image,
        "true": labels[true_idx],
        "pred": labels[pred_idx],
        "status": status
    })

# Render 6 random predictions from the evaluation subset
selected = random.sample(samples, 6)
for i, s in enumerate(selected):
    plt.figure(figsize=(3,3))
    plt.imshow(s["image"])
    plt.title(f"Sample {i+1}\n{s['status']}\nTrue: {s['true']}\nPred: {s['pred']}", fontsize=9)
    plt.axis("off")
    plt.show()
    print(f"Sample {i+1} | {s['status']} | True: {s['true']} | Predicted: {s['pred']}\n")